# Stage 2: Face Recognition & Feature Embedding Extraction
Demonstrating 512-dimensional $L_2$-normalized feature vector extraction and Cosine Similarity matching for **DynaCampus-FR**.

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.alignment import FaceAligner
from src.recognition import FaceEmbedder

print("✅ Successfully imported FaceAligner and FaceEmbedder!")

## 1. Initialize FaceEmbedder
Initializing the embedder (with `mock=True` fallback enabled for offline verification).

In [ ]:
embedder = FaceEmbedder(mock=True)
print(f"Embedder Model: {embedder.model_name}")
print(f"Embedding Dimension: {embedder.embedding_dim}")

## 2. Extract Feature Embedding from Aligned Face Crop
Creating synthetic aligned face crops and extracting 512-dimensional $L_2$-normalized embeddings.

In [ ]:
# Synthetic Crop A (Identity 1)
crop_a = np.full((112, 112, 3), 150, dtype=np.uint8)
cv2.circle(crop_a, (35, 45), 8, (20, 20, 20), -1)
cv2.circle(crop_a, (77, 45), 8, (20, 20, 20), -1)

# Synthetic Crop B (Identity 1 - slight brightness variation)
crop_b = np.full((112, 112, 3), 155, dtype=np.uint8)
cv2.circle(crop_b, (35, 45), 8, (20, 20, 20), -1)
cv2.circle(crop_b, (77, 45), 8, (20, 20, 20), -1)

# Synthetic Crop C (Identity 2 - different features)
crop_c = np.full((112, 112, 3), 80, dtype=np.uint8)
cv2.rectangle(crop_c, (20, 30), (50, 60), (200, 200, 200), -1)
cv2.rectangle(crop_c, (62, 30), (92, 60), (200, 200, 200), -1)

# Extract embeddings
emb_a = embedder.extract_embedding(crop_a)
emb_b = embedder.extract_embedding(crop_b)
emb_c = embedder.extract_embedding(crop_c)

print(f"Embedding A shape: {emb_a.shape}, L2 Norm: {np.linalg.norm(emb_a):.4f}")
print(f"Embedding B shape: {emb_b.shape}, L2 Norm: {np.linalg.norm(emb_b):.4f}")
print(f"Embedding C shape: {emb_c.shape}, L2 Norm: {np.linalg.norm(emb_c):.4f}")

## 3. Cosine Similarity Matrix Computation
Computing cosine similarity between same-identity and different-identity embeddings.

In [ ]:
sim_a_b = embedder.compute_similarity(emb_a, emb_b)
sim_a_c = embedder.compute_similarity(emb_a, emb_c)

print(f"Cosine Similarity (Crop A vs Crop B - Same ID): {sim_a_b:.4f}")
print(f"Cosine Similarity (Crop A vs Crop C - Diff ID): {sim_a_c:.4f}")

batch = np.stack([crop_a, crop_b, crop_c])
batch_embs = embedder.extract_embedding(batch)
sim_matrix = embedder.compute_similarity_matrix(batch_embs, batch_embs)

plt.figure(figsize=(6, 5))
plt.imshow(sim_matrix, cmap="viridis", vmin=-1.0, vmax=1.0)
plt.colorbar(label="Cosine Similarity")
plt.xticks([0, 1, 2], ["Crop A", "Crop B", "Crop C"])
plt.yticks([0, 1, 2], ["Crop A", "Crop B", "Crop C"])
plt.title("Pairwise Cosine Similarity Matrix")
for i in range(3):
    for j in range(3):
        plt.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", color="white" if sim_matrix[i, j] < 0.5 else "black")
plt.tight_layout()
plt.show()